Tunable TIA design example with the optimizer orchestrator class (SPICE-based optimization)

# Pre-body

## Clearing past runs (optional)

In [6]:
!rm -rf logs/ # clear logs
!rm -rf spice_out/

335.80s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
341.36s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


## IIC-OSIC Env Setup

In [7]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice


351.92s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


## Library Imports

In [13]:
import logging

from pathlib import Path

from symxplorer.optimization    import Circuit_Optimizer_Orchestrator_with_SPICE, Optimizer_Type_Enum
from symxplorer.logging         import setup_loggers

setup_loggers()
logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

05:02:11 - SymXplorer: [INFO] 🚀 Logger initialized and ready!
05:02:11 - SymXplorer: [INFO] 📄 Log file: /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/sizing/logs/SymXplorer_2025-10-15_05-02-11.log
05:02:11 - SymXplorer: [INFO] 🔧 spicelib logger set to 50
05:02:11 - SymXplorer.jupyter: [INFO] Spicelib_Wrapper imported successfully.


# Instantiations


In [14]:
# ----------------------------
# Instantiations
# ----------------------------
ws_root = "/foss/designs/eda/SymXplorer/examples/tunable-tia"
pdk_name = "ihp-sg13g2"
yaml_file_name = "tia-topo-1/project_setup.yaml"

project_setup_path = f"{ws_root}/{pdk_name}/spice/{yaml_file_name}"

orchestrator = Circuit_Optimizer_Orchestrator_with_SPICE(
    project_setup_path=project_setup_path,
    optimizer_type=Optimizer_Type_Enum.AX_CONSTRAINT,
    verbose=False
)


05:02:16 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=250, random_seed=48
05:02:16 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
05:02:16 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
05:02:16 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
05:02:16 - SymXplorer.domains: [INFO] 	Number of target specs: 4
05:02:16 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=100e6, range=1.00e+08 tolerance=10000000.0, goal=exact, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=10.0, enable=True, description=Center frequency)
05:02:16 - SymXplorer.domains: [INFO] 		- TargetSpec(name=q, target=10, range=1.00e+01 tolerance=2, goal=exact, sim_type=ac, enable=False, error_type=relative-absolute, weight=10.0, enable=False, description=quality factor)
05:02:16 - SymXplorer.domains: [INFO] 		- TargetSpec(name=ga

In [15]:
orchestrator.run_sanity_on_spicelib_wrapper()

05:02:16 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
05:02:16 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
05:02:16 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/sizing/spice_out/sanity_check/Tunable-TIA-1_sanity.log
05:02:16 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/sizing/spice_out/sanity_check/Tunable-TIA-1_sanity.raw
05:02:16 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
05:02:16 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

In [16]:
circuit_optimizer = orchestrator.get_optimizer()
type(circuit_optimizer)

05:02:16 - SymXplorer.orchestrator: [INFO] creating the circuit_optimizer of type ax_constraint
05:02:16 - SymXplorer.base_optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 4 target specs
05:02:16 - SymXplorer.Ax: [INFO] started the <class 'symxplorer.optimization.bayesian_ax.Ax_Spice_Constraint_Satisfaction'> optimizer class
05:02:16 - SymXplorer.orchestrator: [INFO] created the circuit_optimizer; type <class 'symxplorer.optimization.bayesian_ax.Ax_Spice_Constraint_Satisfaction'>


symxplorer.optimization.bayesian_ax.Ax_Spice_Constraint_Satisfaction

# Main Body

## Optimization

In [17]:
circuit_optimizer.parameterize()

[RangeParameterConfig(name='x_dut_nfet_w', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_nfet_l', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_cap_w', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_cap_l', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_res_s_l', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_res_s_w', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_res_3_l', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_res_3_w', bounds=(0, 100), parameter_type='float', step_size=None, scaling=None),
 RangeParameterConfig(name='x_dut_ind_size', bounds=(0, 100), parameter_type='

In [18]:
circuit_optimizer.optimize()

05:02:44 - SymXplorer.base_optimizer: [INFO] Optimization process started.
[WARNING 10-15 05:02:44] ax.api.client: Metric IMetric('fc') not found in optimization config, added as tracking metric.
[WARNING 10-15 05:02:44] ax.api.client: Metric IMetric('q') not found in optimization config, added as tracking metric.
[WARNING 10-15 05:02:44] ax.api.client: Metric IMetric('gain_db') not found in optimization config, added as tracking metric.
[WARNING 10-15 05:02:44] ax.api.client: Metric IMetric('pm') not found in optimization config, added as tracking metric.
Optimizing:   0%|          | 0/250 [00:00<?, ?trial/s][INFO 10-15 05:02:44] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol'), GenerationNode(node_name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, model_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')]), GenerationNode(node_name='MBM', generator_s

ValueError: Non-finite data found for metric pm: nan

In [ ]:
circuit_optimizer.plot_score(save_path=project_setup_yaml.parent / "loss_curve.html", show=True)

## Inspection & Visualization

### (1) Best Param

In [ ]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
metadata

In [ ]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param] :0.2e}")

In [ ]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

### (3) Metric Trace

In [ ]:
circuit_optimizer.plot_optimization_trace(metric_x='pm', metric_y='gain_db', show=True)

In [ ]:
circuit_optimizer.plot_score_value_by_spec(spec_name="gain_db", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="fc", show=True)

### (4) Design Space Exploration

In [ ]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_w", param_y="x_dut_nfet_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_cap_l", param_y="x_dut_cap_w", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="vbias", param_y="x_dut_ind_size", show=True)